# Understanding ONNX | Part 2 — Inference only

**One trained model, multiple execution paths.** Begin with the artifacts from Part 1. No dataset is loaded for training, no estimator is fitted, and no model is exported here.

In [ ]:
from pathlib import Path

# Find the repository whether Jupyter starts in the root or this part's folder.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data" / "temperature_conversion.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the tutorial repository.")
MODELS = ROOT / "models"
JOBLIB_PATH = MODELS / "linear_regression.joblib"
ONNX_PATH = MODELS / "linear_regression.onnx"

## 1. Prepare identical sample inputs
Each row represents one Celsius temperature. ONNX expects float32; the Joblib baseline uses float64 as in training.

In [ ]:
import numpy as np
import onnxruntime as ort
from time import perf_counter

if not ONNX_PATH.is_file() or not JOBLIB_PATH.is_file():
    raise FileNotFoundError("Create both artifacts in Part 1 before running this tutorial.")
samples = np.array([[0.0], [25.0], [100.0]], dtype=np.float32)
print("Available providers:", ort.get_available_providers())

## 2. Run the original model on CPU
Joblib is a serialization format, not an inference engine. Scikit-learn performs the prediction.

In [ ]:
import joblib

# Load only the trusted artifact created in Part 1; no fit() call is needed.
model = joblib.load(JOBLIB_PATH)
joblib_predictions = model.predict(samples.astype(np.float64))
print("Joblib CPU:", joblib_predictions)

## 3. Execute ONNX on CPU
ONNX Runtime loads the graph, discovers its input/output contract and executes the graph using a CPU execution provider. Compare predictions using a tolerance because float32 rounding can differ.

In [ ]:
cpu_session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

def predict(session, batch):
    input_meta = session.get_inputs()[0]
    output_meta = session.get_outputs()[0]
    return session.run([output_meta.name], {input_meta.name: batch})[0].reshape(-1)

for item in cpu_session.get_inputs():
    print("Input contract:", item.name, item.shape, item.type)
cpu_predictions = predict(cpu_session, samples)
print("ONNX CPU:", cpu_predictions)
np.testing.assert_allclose(cpu_predictions, joblib_predictions, rtol=1e-5, atol=1e-4)

## 4. Request NVIDIA CUDA execution
The GPU-enabled runtime and compatible CUDA/cuDNN libraries are required. Provider registration does not prove every operator runs on GPU. Unsupported operators may be assigned to CPU. Use ONNX Runtime profiling to verify actual node placement; this small linear model may run entirely on CPU.

In [ ]:
gpu_session = None
if "CUDAExecutionProvider" not in ort.get_available_providers():
    print("CUDA provider unavailable; GPU experiment skipped.")
else:
    try:
        candidate = ort.InferenceSession(
            str(ONNX_PATH), providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
        )
        if "CUDAExecutionProvider" in candidate.get_providers():
            # Disable Python-level retry after provider failure; graph partitioning
            # may still assign unsupported operators to CPU.
            candidate.disable_fallback()
            gpu_predictions = predict(candidate, samples)
            np.testing.assert_allclose(gpu_predictions, cpu_predictions, rtol=1e-5, atol=1e-4)
            gpu_session = candidate
            print("CUDA-preferred providers:", candidate.get_providers())
            print("CUDA-preferred predictions:", gpu_predictions)
        else:
            print("CUDA initialization fell back to CPU; GPU experiment skipped.")
    except (RuntimeError, OSError, ValueError) as error:
        print("CUDA session failed:", error)

## 5. Compare inference latency
Use identical values and batch sizes, warm-up runs, and repeated synchronous calls. The mean and 95th percentile exclude model loading and include normal API overhead and any host/device transfers. This is not a pure GPU kernel benchmark.

In [ ]:
def benchmark(run, warmup=30, iterations=300):
    for _ in range(warmup):
        run()
    durations = []
    for _ in range(iterations):
        start = perf_counter()
        run()
        durations.append((perf_counter() - start) * 1000)
    return np.mean(durations), np.percentile(durations, 95)

for batch_size in (1, 1024):
    batch = np.linspace(0, 99, batch_size, dtype=np.float32).reshape(-1, 1)
    batch64 = batch.astype(np.float64)
    runners = {"Joblib CPU": lambda: model.predict(batch64),
               "ONNX CPU": lambda: predict(cpu_session, batch)}
    if gpu_session is not None:
        runners["ONNX CUDA-preferred"] = lambda: predict(gpu_session, batch)
    for name, run in runners.items():
        mean, p95 = benchmark(run)
        print(f"{batch_size:4d} | {name:22s} | mean={mean:.4f} ms | p95={p95:.4f} ms")

## What did we learn?
The same ONNX file can be consumed by different execution providers. The tiny graph may be faster on CPU; acceleration depends on operator support, workload and transfer costs. A CUDA-preferred result is not proof of GPU execution.